In [ ]:
import ctypes
ctypes.CDLL("libcusparse.so.11") 

import torch
print(torch.cuda.is_available(), torch.version.cuda)

import dgl
print(dgl.__version__)


In [ ]:
import spatialdata as sd
import spatialdata_io as sio
import pandas as pd
import numpy as np

# Load
BI5XO_A1 = sio.xenium("../01_data/20251029__151959__MBSV20251027/output-XETG00285__0079584__BI5XO_A1__20251029__152131/", morphology_focus=False)
BI5XO_B1 = sio.xenium("../01_data/20251029__151959__MBSV20251027/output-XETG00285__0079584__BI5XO_B1__20251029__152132/", morphology_focus=False)
BI5XO_C1 = sio.xenium("../01_data/20251029__151959__MBSV20251027/output-XETG00285__0079584__BI5XO_C1__20251029__152132/", morphology_focus=False)
BI5XO_D1 = sio.xenium("../01_data/20251029__151959__MBSV20251027/output-XETG00285__0079584__BI5XO_D1__20251029__152132/", morphology_focus=False)
BXNFB_A1 = sio.xenium("../01_data/20251029__151959__MBSV20251027/output-XETG00285__0079587__BXNFB_A1__20251029__152132/", morphology_focus=False)
BXNFB_B1 = sio.xenium("../01_data/20251029__151959__MBSV20251027/output-XETG00285__0079587__BNXFB_B1__20251029__152132/", morphology_focus=False)
BXNFB_C1 = sio.xenium("../01_data/20251029__151959__MBSV20251027/output-XETG00285__0079587__BXNFB_C1__20251029__152132/", morphology_focus=False)
BXNFB_D1 = sio.xenium("../01_data/20251029__151959__MBSV20251027/output-XETG00285__0079587__BXNFB_D1__20251029__152132/", morphology_focus=False)

# Concatenate 
atlas_sd = sd.concatenate(
    {
        "BI5XO_A1": BI5XO_A1,
        "BI5XO_B1": BI5XO_B1,
        "BI5XO_C1": BI5XO_C1,
        "BI5XO_D1": BI5XO_D1,
        "BXNFB_A1": BXNFB_A1,
        "BXNFB_B1": BXNFB_B1,
        "BXNFB_C1": BXNFB_C1,
        "BXNFB_D1": BXNFB_D1,
    }
)


In [ ]:
import re
import anndata as ad
sdata = atlas_sd

# 1) collect all tables named like "table-BI5XO_A1"
table_keys = [k for k in sdata.tables.keys() if k.startswith("table-")]
table_keys = sorted(table_keys)
print("Found tables:", table_keys)

adatas = []
for k in table_keys:
    sample_id = k.replace("table-", "")  # e.g. BI5XO_A1
    
    a = sdata.tables[k].copy()
    a.obs["sample_id"] = sample_id

    # 2) determine the per-sample cell id string
    # Xenium tables almost always have obs["cell_id"]; if not, fall back to obs_names
    if "cell_id" in a.obs.columns:
        cell = a.obs["cell_id"].astype(str)
    else:
        cell = pd.Index(a.obs_names).astype(str)

    # 3) make global IDs exactly like your annotation CSV: SAMPLE_CELLID
    # e.g. BI5XO_A1_aaaajmod-1
    a.obs["cell_id_global"] = a.obs["sample_id"].astype(str) + "_" + cell

    # Put the global ids as obs_names (so join() works)
    a.obs_names = pd.Index(a.obs["cell_id_global"].values)
    a.obs_names_make_unique()  # should be no-op

    adatas.append(a)

# 4) concatenate all samples into one AnnData
adata = ad.concat(adatas, join="outer", merge="same", label="sample_id_concat", keys=[a.obs["sample_id"][0] for a in adatas])

# After concat, keep/restore sample_id in obs in a stable way
if "sample_id" not in adata.obs.columns:
    # ad.concat often stores the key in obs["sample_id_concat"]
    adata.obs["sample_id"] = adata.obs["sample_id_concat"].astype(str)

print("Merged AnnData:", adata.shape)
print("Samples:", adata.obs["sample_id"].value_counts().to_dict())

In [ ]:
ann = pd.read_csv("../01_data/xenium_annotations.csv", index_col=0)

# join on obs_names (global ids)
adata.obs = adata.obs.join(ann, how="left")

# basic sanity checks
n_total = adata.n_obs
n_annot = adata.obs["annotations_level1"].notna().sum()
print(f"Cells total: {n_total:,}")
print(f"Annotated:   {n_annot:,} ({n_annot/n_total:.1%})")

missing_in_adata = ann.index.difference(adata.obs_names)
print(f"Annotation rows not found in adata: {len(missing_in_adata):,}")
print("Example missing:", list(missing_in_adata[:10]))


Cells total: 158,779
Annotated:   104,311 (65.7%)
Annotation rows not found in adata: 6,605
Example missing: ['BI5XO_E1_aaaedpga-1', 'BI5XO_E1_aabbodbk-1', 'BI5XO_E1_aabbohok-1', 'BI5XO_E1_aacdebfl-1', 'BI5XO_E1_aaceijcg-1', 'BI5XO_E1_aaeiibea-1', 'BI5XO_E1_aafijfkg-1', 'BI5XO_E1_aafjpggp-1', 'BI5XO_E1_aafnefji-1', 'BI5XO_E1_aagfeief-1']


In [6]:
celltype_key = "annotations_level2"   # or level3
sample_key = "sample_id"

adata_sc = adata[adata.obs[celltype_key].notna()].copy()
print("After keeping annotated cells:", adata_sc.n_obs)


After keeping annotated cells: 104311


In [7]:
print("obsm keys:", list(adata_sc.obsm.keys()))
print("obs cols (subset):", [c for c in adata_sc.obs.columns if "x" in c.lower() or "y" in c.lower()][:40])


obsm keys: ['spatial']
obs cols (subset): []


In [ ]:
# 2D coordinates
print("spatial shape:", adata_sc.obsm["spatial"].shape)  # should be (n_cells, 2)

# sample labels
print(adata_sc.obs["sample_id"].value_counts())

# how many cells are annotated at the level you want
celltype_key = "annotations_level2"  # or level3
print(adata_sc.obs[celltype_key].value_counts(dropna=False).head(20))


In [9]:
print(adata_sc.obs["sample_id"].value_counts())


sample_id
BI5XO_A1    31394
BXNFB_C1    19662
BI5XO_C1    18392
BXNFB_D1    11728
BI5XO_D1     9823
BI5XO_B1     8750
BXNFB_B1     3450
BXNFB_A1     1112
Name: count, dtype: int64


In [ ]:
import scanpy as sc
import numpy as np

# Keep original counts in a layer (optional but nice)
if "counts" not in adata_sc.layers:
    adata_sc.layers["counts"] = adata_sc.X.copy()

# Basic filtering (tune as you like)
sc.pp.filter_genes(adata_sc, min_cells=10)

# Normalize + log
sc.pp.normalize_total(adata_sc, target_sum=1e4)
sc.pp.log1p(adata_sc)

# HVGs + scaling + PCA
sc.pp.highly_variable_genes(adata_sc, n_top_genes=2000, flavor="seurat_v3")
adata_sc = adata_sc[:, adata_sc.var["highly_variable"]].copy()

sc.pp.scale(adata_sc, max_value=10)
sc.tl.pca(adata_sc, n_comps=50)


In [ ]:
import scniche as sn

sn.pp.set_seed()

celltype_key = "annotations_level2"   # <- choose
sample_key = "sample_id"
use_rep = "X_pca"

k_cutoff = 30
batch_num = 100
epochs = 100

adata_sc = sn.pp.process_multi_slices(
    adata=adata_sc,
    celltype_key=celltype_key,
    sample_key=sample_key,
    mode="KNN",
    k_cutoff=k_cutoff,
    is_pca=False,
    verbose=False,
    layer_key=use_rep
)

adata_sc = sn.pp.prepare_data_batch(adata=adata_sc, verbose=False, batch_num=batch_num)

model = sn.tr.Runner_batch(adata=adata_sc, device="cuda:0", verbose=False)
adata_sc = model.fit(lr=0.01, epochs=epochs)

# clustering: pick a target number (two common choices below)
adata_sc = sn.tr.clustering(adata=adata_sc, target_k=4)


print(adata_sc.obs["scNiche"].value_counts().head(20))


In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc

plt.rcParams["figure.figsize"] = (4, 4)

one = adata_sc.obs["sample_id"].unique()[0]
sc.pl.embedding(
    adata_sc[adata_sc.obs["sample_id"] == one],
    basis="spatial",
    color=[celltype_key, "scNiche"],
    s=8
)
one = adata_sc.obs["sample_id"].unique()[1]
sc.pl.embedding(
    adata_sc[adata_sc.obs["sample_id"] == one],
    basis="spatial",
    color=[celltype_key, "scNiche"],
    s=8
)
one = adata_sc.obs["sample_id"].unique()[2]
sc.pl.embedding(
    adata_sc[adata_sc.obs["sample_id"] == one],
    basis="spatial",
    color=[celltype_key, "scNiche"],
    s=8
)
one = adata_sc.obs["sample_id"].unique()[3]
sc.pl.embedding(
    adata_sc[adata_sc.obs["sample_id"] == one],
    basis="spatial",
    color=[celltype_key, "scNiche"],
    s=8
)


In [ ]:
# cell type enrichment
sn.al.enrichment(adata_sc, id_key=celltype_key, val_key='scNiche', library_key='sample_id')

# plot
kwargs = {'figsize': (8, 8), 'vmax': 4, 'cmap': 'YlOrBr', 'linewidths': 0, 'linecolor': 'white', }
categories1 = adata_sc.obs['scNiche'].cat.categories
sn.pl.enrichment_heatmap(adata=adata_sc, id_key=celltype_key, val_key='scNiche', binarized=False, show_pval=True, 
                         anno_key=None, kwargs=kwargs)


In [32]:
adata_sc.uns['scNiche_annotations_level2_pval'].to_csv("../03_output/scNiche_annotations_level2_pval.csv")
